## Imports

In [4]:
import pandas as pd
import numpy as np
from typing import Any

import os

from tqdm import tqdm
import optuna
from lightfm import LightFM
from rectools.columns import Columns
from rectools.dataset import Dataset
from rectools.models import LightFMWrapperModel
import polars as pl

import json

## Constants

In [5]:
K = 20
MIN_WATCH_TIME = 60

user_column = "user_id"
item_column = "item_id"
event_type_column = "event_type"
watch_time_column = "watch_time"
date_column = "date"

In [6]:
def seed_everything(seed: int = 334791) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything()

## Data

In [11]:
def split_data(data: pd.DataFrame, column: str, splt_value: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_data = data[data[column].dt.date < splt_value]
    test_data = data[data[column].dt.date  >= splt_value]
    return train_data, test_data

dataset = pl.read_parquet("data/train.parquet").to_pandas()
train, val = split_data(dataset, date_column, pd.to_datetime("2024-12-02").date())

target_users = pl.read_parquet("data/target_user_ids.parquet").to_pandas()
target_user_ids = target_users["user_id"].values

How many items might be deleted

In [16]:
def check_coverage_after_item_filter(data: pd.DataFrame, min_value: int, users: np.ndarray) -> dict[str, int | float]:
    results = {}

    item_counts = data[item_column].value_counts()
    kept_items  = set(item_counts[item_counts > min_value].index)
    filtered    = data[data[item_column].isin(kept_items)]
    users_with_items = set(filtered[user_column].unique())
    n_empty = sum(1 for u in target_users if u not in users_with_items)
    results = {
        "items_kept": len(kept_items),
        "items_total": len(item_counts),
        "target_users_with_no_items": n_empty,
        "empty_percent": 100 * n_empty / len(target_users),
    }

    return results

In [17]:
check_coverage_after_item_filter(dataset, 150, target_user_ids)

{'items_kept': 56346,
 'items_total': 1884722,
 'target_users_with_no_items': 1,
 'empty_percent': 0.0004996202885806787}

In [19]:
item_stats = dataset[item_column].value_counts()
allowed_items = set(item_stats[item_stats > 150].index.tolist())

new_train = train[train[item_column].isin(allowed_items)]
new_train.shape, train.shape, len(allowed_items)

((18541832, 5), (38447537, 5), 56346)

In [20]:
train = new_train
val = val[(val[watch_time_column] > MIN_WATCH_TIME) | (val[event_type_column] != "watch_time")]

## Datasets

In [21]:
def build_interactions_like(df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        df[df[event_type_column] == "like"]
        .groupby([user_column, item_column])
        .agg(cnt=(event_type_column, "size"), date=(date_column, "max"))
        .reset_index()
    )
    if agg.empty:
        return agg
    agg[Columns.Weight] = agg["cnt"]
    return agg[[user_column, item_column, Columns.Weight, date_column]].rename(columns={
        user_column: Columns.User, item_column: Columns.Item, date_column: Columns.Datetime,
    })


def build_interactions_favorite(df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        df[df[event_type_column] == "favorite"]
        .groupby([user_column, item_column])
        .agg(cnt=("event_type", "size"), date=(date_column, "max"))
        .reset_index()
    )
    if agg.empty:
        return agg
    agg[Columns.Weight] = agg["cnt"]
    return agg[[user_column, item_column, Columns.Weight, date_column]].rename(columns={
        user_column: Columns.User, item_column: Columns.Item, date_column: Columns.Datetime,
    })


def build_interactions_watchtime(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df[(df[event_type_column] == "watch_time") & (df["watch_time"] > MIN_WATCH_TIME)]
    agg = (
        filtered
        .groupby([user_column, item_column])
        .agg(wt_sum=("watch_time", "sum"), date=(date_column, "max"))
        .reset_index()
    )
    agg[Columns.Weight] = np.log(agg["wt_sum"] + 1)
    return agg[[user_column, item_column, Columns.Weight, date_column]].rename(columns={
        user_column: Columns.User, item_column: Columns.Item, date_column: Columns.Datetime,
    })

## Model and recs

In [28]:
def train_model(
    interactions: pd.DataFrame,
    no_components: int,
    loss: str,
    learning_rate: float,
    item_alpha: float,
    user_alpha: float,
    epochs: int,
) -> tuple[LightFMWrapperModel, Dataset]:
    dataset = Dataset.construct(interactions)
    lfm = LightFM(
        no_components=no_components,
        loss=loss,
        learning_rate=learning_rate,
        item_alpha=item_alpha,
        user_alpha=user_alpha,
        random_state=42,
    )
    model = LightFMWrapperModel(model=lfm, epochs=epochs, num_threads=32)
    model.fit(dataset)
    return model, dataset


def get_recs(
    model: LightFMWrapperModel,
    dataset: Dataset,
    users: list[int],
    k: int,
    filter_viewed: bool =True
) -> dict[int, list[int]]:
    known = set(dataset.user_id_map.external_ids)
    valid_users = [u for u in users if u in known]
    reco_df = model.recommend(
        users=valid_users,
        dataset=dataset,
        k=k,
        filter_viewed=filter_viewed,
    )
    return (
        reco_df.sort_values(Columns.Rank)
        .groupby(Columns.User)[Columns.Item]
        .apply(list)
        .to_dict()
    )


def get_ground_truth(data: pd.DataFrame) -> dict[int, set[int]]:
    return data.groupby(user_column)[item_column].apply(set).to_dict()


def precision_at_k(data: dict[int, set[int]], ground_truth: dict[int, set[int]], k: int = K) -> float:
    precisions = []
    for user, rec_items in recs.items():
        relevant = ground_truth.get(user, set())
        if not relevant:
            continue
        hits = len(set(rec_items[:k]) & relevant)
        precisions.append(hits / k)
    return float(np.mean(precisions)) if precisions else 0.0

## Tune

In [ ]:
def tune_single_model(interactions: pd.DataFrame, val_users: list[int], val_gt: dict[int, set[int]], name: str, n_trials: int) -> tuple[pd.DataFrame, dict[str, Any]]:
    results = []

    def objective(trial):
        no_components = trial.suggest_categorical("no_components", [16, 32, 64, 128, 256, 512, 750, 1024])
        loss = trial.suggest_categorical("loss", ["warp"])
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 0.5, log=True)
        item_alpha = trial.suggest_float("item_alpha", 1e-6, 1e-1, log=True)
        user_alpha = trial.suggest_float("user_alpha", 1e-6, 1e-1, log=True)
        epochs = trial.suggest_int("epochs", 5, 30)

        model, dataset = train_model(
            interactions, no_components, loss, learning_rate,
            item_alpha, user_alpha, epochs,
        )
        recs = get_recs(model, dataset, val_users, k=K)
        p20  = precision_at_k(recs, val_gt)

        results.append({
            "no_components": no_components,
            "loss": loss,
            "learning_rate": learning_rate,
            "item_alpha": item_alpha,
            "user_alpha": user_alpha,
            "epochs": epochs,
            "precision@20": p20,
        })

        results_df = pd.DataFrame(results).sort_values("precision@20", ascending=False).reset_index(drop=True)
        results_df.to_csv(f"logs/{name}.csv")
        
        return p20

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    results_df = pd.DataFrame(results).sort_values("precision@20", ascending=False).reset_index(drop=True)
    
    return results_df, study.best_params

In [30]:
val_gt = get_ground_truth(val)
val_users = list(val_gt.keys())

int_like = build_interactions_like(train)
int_fav = build_interactions_favorite(train)
int_wt = build_interactions_watchtime(train)

In [ ]:
_, best_like = tune_single_model(int_like, val_users, val_gt, "like", 70)
_, best_fav  = tune_single_model(int_fav,  val_users, val_gt, "favorite", 70)
_, best_wt   = tune_single_model(int_wt,   val_users, val_gt, "watchtime", 70)

[I 2026-04-07 03:38:07,177] A new study created in memory with name: no-name-f9a1bb63-0b38-491c-b732-a846cf4d074a



  Tuning like model (70 trials)...


  0%|          | 0/70 [00:00<?, ?it/s]

[I 2026-04-07 03:38:32,201] Trial 0 finished with value: 6.536924283739867e-05 and parameters: {'no_components': 1024, 'loss': 'warp', 'learning_rate': 0.1654918545625831, 'item_alpha': 0.006348961154614583, 'user_alpha': 5.3835563602456485e-06, 'epochs': 14}. Best is trial 0 with value: 6.536924283739867e-05.


In [ ]:
with open(f"logs/metrics_best_like.json", "w") as f:
    json.dump(best_like, f)
with open(f"logs/metrics_best_fav.json", "w") as f:
    json.dump(best_fav, f)
with open(f"logs/metrics_best_wt.json", "w") as f:
    json.dump(best_wt, f)

## The best setup

In [28]:
model_like, dataset_like = train_model(int_like, **best_like)
model_fav, dataset_fav = train_model(int_fav, **best_fav)
model_wt, dataset_wt = train_model(int_wt, **best_wt)

In [29]:
fetch_k = K + 20
recs_like_val = get_recs(model_like, dataset_like, val_users, fetch_k)
recs_fav_val  = get_recs(model_fav, dataset_fav, val_users, fetch_k)
recs_wt_val   = get_recs(model_wt, dataset_wt, val_users, fetch_k)

In [30]:
len(recs_like_val)

26771

In [31]:
len(recs_like_val)

26771

In [32]:
len(recs_fav_val)

18743

In [33]:
precision_at_k(recs_like_val, val_gt)

0.005212730193119421

In [34]:
precision_at_k(recs_fav_val, val_gt)

0.006090273702182149

In [35]:
precision_at_k(recs_wt_val, val_gt)

0.0074914660794939405

In [36]:
def merge_recs(recs_like: dict[int, list[int]], recs_fav: dict[int, list[int]], recs_wt: dict[int, list[int]], users: list[int]) -> dict[int, list[int]]:
    result = {}
    for user in users:
        seen   = set()
        merged = []
        for recs in [recs_fav, recs_like, recs_wt]:
            for item in recs.get(user, []):
                if len(merged) >= K:
                    break
                if item not in seen:
                    merged.append(item)
                    seen.add(item)
            if len(merged) >= K:
                break
        result[user] = merged
    return result

In [37]:
total_recs = merge_recs(recs_like_val, recs_fav_val,recs_wt_val, val_users)

In [38]:
precision_at_k(total_recs, val_gt)

0.0069621530426499725

## Submission inference

In [111]:
val = val[val[item_column].isin(allowed_items)]
total_dataset = pd.concat([train, val])

In [116]:
total_dataset["date"].dt.date.value_counts()

date
2024-11-28    4631997
2024-11-29    4343853
2024-11-30    3413941
2024-12-01    3158288
2024-11-27    2993753
2024-12-02    2209846
2024-12-03     585521
Name: count, dtype: int64

In [117]:
int_like = build_interactions_like(total_dataset)
int_fav = build_interactions_favorite(total_dataset)
int_wt = build_interactions_watchtime(total_dataset)

In [118]:
model_like_target, dataset_like_target = train_model(int_like, **best_like)
model_fav_target, dataset_fav_target = train_model(int_fav, **best_fav)
model_wt_target, dataset_wt_target = train_model(int_wt, **best_wt)

In [120]:
recs_like_target = get_recs(model_like_target, dataset_like_target, target_user_ids, fetch_k)
recs_fav_target = get_recs(model_fav_target, dataset_fav_target, target_user_ids, fetch_k)
recs_wt_target = get_recs(model_wt_target, dataset_wt_target, target_user_ids, fetch_k)

In [121]:
total_recs = merge_recs(recs_like_target, recs_fav_target, recs_wt_target, target_user_ids)
target_users["item_ids"] = target_users[user_column].map(total_recs)

In [123]:
target_users[target_users["item_ids"].str.len() != 20]

,user_id,item_ids
23,10890074385859229523,[]
113,13289378714338971969,[]
127,2472594548546584806,[]
266,407800405085052423,[]
291,12161678876707880832,[]
...,...,...
199535,2092593239272682907,[]
199577,4768694889412472367,[]
199670,1383438564261662052,[]
199730,18278866565743649388,[]


In [93]:
filtered = total_dataset[(total_dataset[watch_time_column] > 60) | (total_dataset[event_type_column] != "watch_time")]
top_items = list(filtered[item_column].value_counts()[:20].index)

In [126]:
target_users["item_ids"] = target_users["item_ids"].apply(lambda x: str(top_items) if x == [] else str(x))

In [128]:
target_users

,user_id,item_ids
0,16168876412835480816,"[42145, 31241, 2982, 3020, 92, 23076, 88039, 1..."
1,641347240838295768,"[13240, 26137, 51941, 860, 28113, 121421, 1535..."
2,11282462919885101133,"[86064, 35079, 40572, 3524, 85546, 35470, 3139..."
3,10187162321047637271,"[17746, 2655, 41088, 19827, 6964, 77794, 59512..."
4,1522284854545669005,"[4856, 16704, 39558, 5749, 10154, 48854, 37086..."
...,...,...
200147,15721614028231690371,"[5068, 4916, 5502, 2655, 7190, 6980, 1907, 401..."
200148,13933216566018619926,"[25483, 12644, 29327, 2689, 127719, 2783, 1884..."
200149,9131592974022462243,"[24003, 22750, 19328, 7417, 6856, 5780, 12029,..."
200150,18270889459511706192,"[16048, 2078, 199168, 149433, 39316, 29550, 44..."


In [130]:
target_users[target_users["item_ids"] == str(top_items)]

,user_id,item_ids
23,10890074385859229523,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
113,13289378714338971969,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
127,2472594548546584806,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
266,407800405085052423,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
291,12161678876707880832,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
...,...,...
199535,2092593239272682907,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
199577,4768694889412472367,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
199670,1383438564261662052,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."
199730,18278866565743649388,"[5068, 7440, 2655, 5502, 17746, 4856, 4916, 71..."


In [131]:
target_users.to_csv("sub2.csv", index=False)